# Part 14: Record Store and CID Addressing

This notebook builds a simplified record store with content-addressed storage. Records are the core
data unit in ATProto — each lives in a collection under a record key, addressed by a Content
Identifier (CID).

**What you'll learn:**

- How CIDs provide content-addressed identity
- Record CRUD with AT URI keys
- How commits bundle record writes

**Estimated Time:** 20-25 minutes

## Step 1: CID Helper

CIDs are content identifiers — a hash of the data that serves as its address. In real
implementations, `CID.m` uses SHA-256 and base32 encoding. Here we use a simplified XOR-fold hash to
demonstrate the concept: same data always produces the same CID.

In [ ]:
@interface CIDHelper : NSObject
- (NSString *)cidForData:(NSData *)data;
- (BOOL)verifyData:(NSData *)data cid:(NSString *)cid;
@end

@implementation CIDHelper
- (NSString *)cidForData:(NSData *)data {
    // Simplified hash: XOR-fold bytes into 4 bytes, then hex-encode
    // Real implementations uses SHA-256 → base32 (CIDv1 with dag-cbor codec)
    int a = 0;
    int b = 0;
    int c = 0;
    int d = 0;
    const char *bytes = [data bytes];
    int len = [data length];
    for (int i = 0; i < len; i++) {
        a = a ^ bytes[i];
        b = b ^ (bytes[i] << 1);
        c = c ^ (bytes[i] << 2);
        d = d ^ (bytes[i] << 3);
    }
    // Keep only lower byte of each
    a = a & 0xFF;
    b = b & 0xFF;
    c = c & 0xFF;
    d = d & 0xFF;
    return [NSString stringWithFormat:@"bafyrei%02x%02x%02x%02x", a, b, c, d];
}
- (BOOL)verifyData:(NSData *)data cid:(NSString *)cid {
    NSString *computed = [self cidForData:data];
    return [computed isEqualToString:cid];
}
@end

CIDHelper *cidHelper = [[CIDHelper alloc] init];
NSData *hello = [NSData dataWithBytes:"hello" length:5];
NSString *cid = [cidHelper cidForData:hello];
NSLog(@"CID for 'hello': %@", cid);

// Same data produces same CID
NSData *hello2 = [NSData dataWithBytes:"hello" length:5];
NSString *cid2 = [cidHelper cidForData:hello2];
NSLog(@"Same CID: %d", [cid isEqualToString:cid2]);

// Different data produces different CID
NSData *world = [NSData dataWithBytes:"world" length:5];
NSString *cid3 = [cidHelper cidForData:world];
NSLog(@"Different CID: %d", ![cid isEqualToString:cid3]);

// Verification
NSLog(@"Verify correct: %d", [cidHelper verifyData:hello cid:cid]);
NSLog(@"Verify wrong: %d", [cidHelper verifyData:world cid:cid]);

## Step 2: Record Store

Records are stored by AT URI: `at://did/collection/rkey`. services provides CRUD operations that
delegate to the repository.

In [ ]:
@interface RecordStore : NSObject
@property (nonatomic, strong) NSMutableDictionary *records;
@property (nonatomic, strong) CIDHelper *cidHelper;
- (NSString *)putRecord:(NSDictionary *)record collection:(NSString *)collection rkey:(NSString *)rkey forDid:(NSString *)did;
- (NSDictionary *)getRecord:(NSString *)collection rkey:(NSString *)rkey forDid:(NSString *)did;
- (BOOL)deleteRecord:(NSString *)collection rkey:(NSString *)rkey forDid:(NSString *)did;
@end

@implementation RecordStore
- (instancetype)init {
    self = [super init];
    if (self) {
        _records = [NSMutableDictionary dictionary];
        _cidHelper = [[CIDHelper alloc] init];
    }
    return self;
}
- (NSString *)atUriForCollection:(NSString *)collection rkey:(NSString *)rkey did:(NSString *)did {
    return [NSString stringWithFormat:@"at://%@/%@/%@", did, collection, rkey];
}
- (NSString *)putRecord:(NSDictionary *)record collection:(NSString *)collection rkey:(NSString *)rkey forDid:(NSString *)did {
    NSString *uri = [self atUriForCollection:collection rkey:rkey did:did];
    // Compute CID from record description (simplified)
    NSString *desc = [record description];
    NSData *descData = [NSData dataWithBytes:[desc UTF8String] length:[desc length]];
    NSString *cid = [_cidHelper cidForData:descData];
    NSMutableDictionary *stored = [NSMutableDictionary dictionaryWithDictionary:record];
    [stored setObject:cid forKey:@"cid"];
    [_records setObject:stored forKey:uri];
    NSLog(@"Put %@ -> %@", uri, cid);
    return cid;
}
- (NSDictionary *)getRecord:(NSString *)collection rkey:(NSString *)rkey forDid:(NSString *)did {
    NSString *uri = [self atUriForCollection:collection rkey:rkey did:did];
    NSDictionary *record = [_records objectForKey:uri];
    if (record != nil) {
        NSLog(@"Got %@", uri);
    } else {
        NSLog(@"Not found: %@", uri);
    }
    return record;
}
- (BOOL)deleteRecord:(NSString *)collection rkey:(NSString *)rkey forDid:(NSString *)did {
    NSString *uri = [self atUriForCollection:collection rkey:rkey did:did];
    if ([_records objectForKey:uri] == nil) return NO;
    [_records removeObjectForKey:uri];
    NSLog(@"Deleted %@", uri);
    return YES;
}
@end

## Step 3: Commit Builder

When records are written, the repository creates a commit. A commit bundles the writes with the
previous root CID, a revision number, and the new data root CID. This mirrors `RepoCommit` in the
real codebase.

In [ ]:
@interface CommitBuilder : NSObject
@property (nonatomic, assign) int revCounter;
@property (nonatomic, strong) CIDHelper *cidHelper;
- (NSDictionary *)buildCommitForDid:(NSString *)did
                            writes:(NSArray *)writes
                          prevCid:(NSString *)prevCid;
@end

@implementation CommitBuilder
- (instancetype)init {
    self = [super init];
    if (self) {
        _revCounter = 1;
        _cidHelper = [[CIDHelper alloc] init];
    }
    return self;
}
- (NSDictionary *)buildCommitForDid:(NSString *)did
                            writes:(NSArray *)writes
                          prevCid:(NSString *)prevCid {
    NSString *rev = [NSString stringWithFormat:@"%d", _revCounter];
    _revCounter = _revCounter + 1;
    // Compute data CID from writes
    NSString *writesDesc = [writes description];
    NSData *data = [NSData dataWithBytes:[writesDesc UTF8String] length:[writesDesc length]];
    NSString *dataCid = [_cidHelper cidForData:data];
    NSDictionary *commit = @{
        @"did": did,
        @"rev": rev,
        @"prev": (prevCid != nil) ? prevCid : @"",
        @"data": dataCid,
        @"writes": writes
    };
    NSLog(@"Commit rev=%@ data=%@", rev, dataCid);
    return commit;
}
@end

## Step 4: Full Write Path

Wire it all together: create records, generate a commit, verify CID integrity. This shows the write
path: record → store → CID → commit.

In [ ]:
RecordStore *store = [[RecordStore alloc] init];
CommitBuilder *builder = [[CommitBuilder alloc] init];

NSString *did = @"did:plc:abc123";

// Create records
NSString *cid1 = [store putRecord:@{@"text": @"Hello ATProto!", @"createdAt": @"2025-01-01"}
                       collection:@"app.bsky.feed.post"
                             rkey:@"3kabc"
                          forDid:did];

NSString *cid2 = [store putRecord:@{@"displayName": @"Alice", @"description": @"Test user"}
                       collection:@"app.bsky.actor.profile"
                             rkey:@"self"
                          forDid:did];

// Read back
NSDictionary *post = [store getRecord:@"app.bsky.feed.post" rkey:@"3kabc" forDid:did];
NSLog(@"Post CID: %@", [post objectForKey:@"cid"]);

// Build commit
NSArray *writes = @[
    @{@"action": @"create", @"collection": @"app.bsky.feed.post", @"rkey": @"3kabc", @"cid": cid1},
    @{@"action": @"create", @"collection": @"app.bsky.actor.profile", @"rkey": @"self", @"cid": cid2}
];
NSDictionary *commit = [builder buildCommitForDid:did writes:writes prevCid:nil];
NSLog(@"Commit: rev=%@ prev=%@", [commit objectForKey:@"rev"], [commit objectForKey:@"prev"]);

// Delete a record
[store deleteRecord:@"app.bsky.feed.post" rkey:@"3kabc" forDid:did];
NSDictionary *gone = [store getRecord:@"app.bsky.feed.post" rkey:@"3kabc" forDid:did];
NSLog(@"After delete: %@", gone);

## Exercise

Add `listRecords:forDid:limit:` to `RecordStore` that returns records in a collection with
pagination. The method should take a collection name, DID, and limit, and return an NSArray of
records.

Hint: iterate over `_records` keys, filter by `at://did/collection/` prefix, collect matching
records, and return up to `limit` entries.

In [ ]:
// Exercise: implement listRecords:forDid:limit:
// Your code here...

## Solution

In [ ]:
// Solution: listRecords filters by collection prefix and limits results
@interface RecordStore2 : NSObject
@property (nonatomic, strong) NSMutableDictionary *records;
- (NSMutableArray *)listRecords:(NSString *)collection forDid:(NSString *)did limit:(int)limit;
@end

@implementation RecordStore2
- (NSMutableArray *)listRecords:(NSString *)collection forDid:(NSString *)did limit:(int)limit {
    NSString *prefix = [NSString stringWithFormat:@"at://%@/%@/", did, collection];
    NSMutableArray *results = [NSMutableArray array];
    for (NSString *key in _records) {
        if ([key hasPrefix:prefix]) {
            [results addObject:[_records objectForKey:key]];
            if ([results count] >= limit) break;
        }
    }
    return results;
}
@end

// Test it
RecordStore2 *store2 = [[RecordStore2 alloc] init];
store2.records = [NSMutableDictionary dictionary];
[store2.records setObject:@{@"text": @"Post 1"} forKey:@"at://did:plc:x/app.bsky.feed.post/a"];
[store2.records setObject:@{@"text": @"Post 2"} forKey:@"at://did:plc:x/app.bsky.feed.post/b"];
[store2.records setObject:@{@"name": @"Profile"} forKey:@"at://did:plc:x/app.bsky.actor.profile/self"];

NSMutableArray *posts = [store2 listRecords:@"app.bsky.feed.post" forDid:@"did:plc:x" limit:10];
NSLog(@"Posts: %d", [posts count]);

NSMutableArray *limited = [store2 listRecords:@"app.bsky.feed.post" forDid:@"did:plc:x" limit:1];
NSLog(@"Limited: %d", [limited count]);

## Next Steps

You have learned about ATProto's core data structures and protocols. Continue to explore how these
concepts apply when building a PDS.

## What to Read Next

You now understand record storage and CID addressing. Next:

- **Part 15: Blob Storage** — content-addressed binary data
- **Part 16: Firehose and Sync** — event streams and repository diffs